# Latent Space Diagnostics for Deep Embedded Clustering (DEC)

This notebook focuses on the representation learned by Deep Embedded Clustering: the **latent space**.

The previous notebook implemented DEC step by step. Here we ask a more interpretive question:

> Once DEC compresses each sample into a vector `z`, which latent dimensions actually help separate the clusters?

We use the classic `sklearn` handwritten digits dataset because it is small, offline, and visually intuitive. The labels are used only after clustering for evaluation; the DEC training itself remains unsupervised.


## Notebook Map

1. Build a clean DEC latent space on handwritten digits.
2. Treat the latent vectors as a small feature table.
3. Score each latent dimension by cluster separation.
4. Visualize cluster profiles in latent coordinates.
5. Run ablation tests using only selected latent dimensions.
6. Summarize what the latent space appears to have learned.

The key idea is that DEC is not only a clustering method. It is also a representation-learning method. The cluster labels are the visible output, but the latent space is where the model organizes the data.


In [ ]:
# Environment guardrails should run before importing heavy numerical libraries.
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMBA_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))

import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

from dec_model import (
    DEVICE,
    LEARNING_RATE,
    encode_dataset,
    initialize_clusters,
    pretrain_autoencoder,
    train_DEC,
)
from latent_space_tools import (
    build_cluster_profile,
    build_latent_feature_table,
    compare_clustering_stages,
    evaluate_latent_subsets,
    load_preprocessed_digits,
    score_latent_dimensions,
)

np.random.seed(42)
torch.manual_seed(42)

print("Using device:", DEVICE)


## Step 1 — Load a Dataset That Can Form a Meaningful Latent Space

A latent space is only interesting if the input data has structure. Random pixels are useful for debugging code, but they do not contain semantic groups.

The digits dataset contains 8×8 grayscale images of handwritten digits. Each image starts as 64 pixel values. After preprocessing, each row is still just a numeric vector, but now the values are scaled to a stable range for neural-network training.


In [ ]:
X_digits_raw, X_digits, y_digits, digits_images, digits_preprocess = load_preprocessed_digits()

print("Raw digits shape:", X_digits_raw.shape)
print("Preprocessed digits shape:", X_digits.shape)
print("Number of digit classes:", len(np.unique(y_digits)))
print("Pixel range after preprocessing:", (float(X_digits.min()), float(X_digits.max())))
print("Preprocessing pipeline:", digits_preprocess)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit_value, ax in enumerate(axes.ravel()):
    first_idx = np.where(y_digits == digit_value)[0][0]
    ax.imshow(digits_images[first_idx], cmap="gray_r")
    ax.set_title(f"Digit {digit_value}")
    ax.axis("off")

plt.suptitle("One Example Image per Digit Class", y=1.02)
plt.tight_layout()
plt.show()


## Step 2 — Train DEC to Create the Latent Space

The encoder maps each input image `x` to a compact latent vector `z`.

For this notebook we use a 10-dimensional latent space. That means every digit image becomes a row with ten learned coordinates:

`z_0, z_1, z_2, ..., z_9`

The dimensions are not manually designed. They are learned by the autoencoder pretraining and then reshaped by the DEC clustering objective.


In [ ]:
LATENT_LAYER_DIMS = [128, 64, 10]
N_CLUSTERS = 10
BATCH_SIZE_DIGITS = 128
PRETRAIN_EPOCHS_DIGITS = 8
FINETUNE_EPOCHS_DIGITS = 12
DEC_EPOCHS_DIGITS = 25

print("Input dimension:", X_digits.shape[1])
print("Encoder dimensions:", LATENT_LAYER_DIMS)
print("Latent dimension:", LATENT_LAYER_DIMS[-1])
print("Number of clusters:", N_CLUSTERS)


In [ ]:
digits_encoder = pretrain_autoencoder(
    X_digits,
    layer_dims=LATENT_LAYER_DIMS,
    batch_size=BATCH_SIZE_DIGITS,
    pretrain_epochs=PRETRAIN_EPOCHS_DIGITS,
    finetune_epochs=FINETUNE_EPOCHS_DIGITS,
    lr=LEARNING_RATE,
    device=DEVICE,
)

digits_Z_init, digits_mu_init, digits_kmeans_labels = initialize_clusters(
    digits_encoder,
    X_digits,
    k=N_CLUSTERS,
    batch_size=BATCH_SIZE_DIGITS,
    device=DEVICE,
)

digits_dec_result = train_DEC(
    digits_encoder,
    X_digits,
    digits_mu_init,
    batch_size=BATCH_SIZE_DIGITS,
    max_epochs=DEC_EPOCHS_DIGITS,
    update_interval=5,
    lr=LEARNING_RATE,
    tol=0.001,
    device=DEVICE,
)

digits_encoder = digits_dec_result.encoder
digits_mu = digits_dec_result.centroids
digits_dec_labels = digits_dec_result.labels
digits_loss_history = digits_dec_result.loss_history
digits_Z_final = encode_dataset(digits_encoder, X_digits, batch_size=BATCH_SIZE_DIGITS, device=DEVICE)

print("Initial latent shape:", digits_Z_init.shape)
print("Final latent shape:", digits_Z_final.shape)


## Step 3 — Check Whether the Latent Space Supports Clustering

Clustering labels are arbitrary. DEC cluster `0` does not automatically mean digit `0`.

To make the result interpretable, we evaluate the cluster assignments after training using labels that DEC never saw during training.

We use:

- **ARI**: adjusted pairwise agreement between clusters and labels.
- **NMI**: shared information between clusters and labels.
- **Hungarian matched accuracy**: the best one-to-one cluster-to-digit mapping.


In [ ]:
stage_to_labels = {
    "KMeans on pretrained latent space": digits_kmeans_labels,
    "DEC-refined latent space": digits_dec_labels,
}
metrics_df, clustering_details = compare_clustering_stages(y_digits, stage_to_labels)

kmeans_details = clustering_details["KMeans on pretrained latent space"]
dec_details = clustering_details["DEC-refined latent space"]

kmeans_acc = kmeans_details["matched_accuracy"]
kmeans_cm = kmeans_details["confusion_matrix"]
cluster_to_label = dec_details["cluster_to_label"]
aligned_labels = dec_details["aligned_labels"]
dec_cm = dec_details["confusion_matrix"]
dec_acc = dec_details["matched_accuracy"]

display(metrics_df.style.format({"ARI": "{:.3f}", "NMI": "{:.3f}", "matched_accuracy": "{:.3f}"}))
print("Cluster -> digit mapping for evaluation:", cluster_to_label)


In [ ]:
digits_Z_init_2d = PCA(n_components=2, random_state=42).fit_transform(digits_Z_init)
digits_Z_final_2d = PCA(n_components=2, random_state=42).fit_transform(digits_Z_final)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(digits_Z_init_2d[:, 0], digits_Z_init_2d[:, 1], c=y_digits, cmap="tab10", s=12, alpha=0.75)
axes[0].set_title("Pretrained Latent Space\nColored by True Digit")
axes[0].set_xlabel("PCA 1")
axes[0].set_ylabel("PCA 2")

axes[1].scatter(digits_Z_final_2d[:, 0], digits_Z_final_2d[:, 1], c=digits_dec_labels, cmap="tab10", s=12, alpha=0.75)
axes[1].set_title("DEC Latent Space\nColored by DEC Cluster")
axes[1].set_xlabel("PCA 1")
axes[1].set_ylabel("PCA 2")

sns.heatmap(dec_cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2])
axes[2].set_title("True Digit vs DEC Cluster")
axes[2].set_xlabel("DEC cluster")
axes[2].set_ylabel("True digit")

plt.tight_layout()
plt.show()


## Step 4 — Turn the Latent Space Into a Feature Table

Now we stop thinking of `Z` as just a neural-network output and start treating it as a small learned dataset.

Each row is one image. Each `z_*` column is one coordinate in the latent space. We also attach:

- the DEC cluster assignment
- the soft-assignment confidence
- the true digit label, only for interpretation

This table is the bridge between deep learning and ordinary feature analysis.


In [ ]:
latent_df, latent_dim_names, latent_Q, latent_confidence, latent_soft_cluster = build_latent_feature_table(
    digits_Z_final,
    digits_mu,
    digits_dec_labels,
    true_labels=y_digits,
)

print("Latent table shape:", latent_df.shape)
print("Mean DEC confidence:", float(latent_confidence.mean()))
display(latent_df.head())


## Step 5 — Score Latent Dimensions by Cluster Separation

A useful latent dimension should separate cluster centers while staying reasonably consistent inside each cluster.

We use a simple score:

`separation_score = between_cluster_variance / within_cluster_variance`

A high score means the coordinate changes across clusters more than it fluctuates within clusters.


In [ ]:
feature_importance, cluster_means = score_latent_dimensions(
    latent_df,
    latent_dim_names,
)

display(
    feature_importance.style.format("{:.4f}").background_gradient(
        subset=["separation_score", "between_cluster_variance", "abs_corr_with_confidence"],
        cmap="viridis",
    )
)


In [ ]:
top_feature_count = min(8, len(feature_importance))
top_latent_dims = feature_importance.head(top_feature_count).index.tolist()
ordered_latent_dims = feature_importance.index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    data=feature_importance.head(top_feature_count).reset_index().rename(columns={"index": "latent_dim"}),
    x="latent_dim",
    y="separation_score",
    ax=axes[0],
    color="#4C78A8",
)
axes[0].set_title("Most Cluster-Separating Latent Dimensions")
axes[0].set_xlabel("Latent dimension")
axes[0].set_ylabel("Between / within variance")
axes[0].tick_params(axis="x", rotation=45)

sns.heatmap(cluster_means[ordered_latent_dims].T, cmap="vlag", center=0, ax=axes[1])
axes[1].set_title("Cluster Centroid Profile by Latent Dimension")
axes[1].set_xlabel("DEC cluster")
axes[1].set_ylabel("Latent dimension")

plt.tight_layout()
plt.show()

print("Top latent dimensions:", top_latent_dims)


## Step 6 — Interpret Cluster Profiles in Latent Coordinates

A cluster profile is the average latent signature of a cluster.

If two clusters have very similar profiles, DEC may struggle to separate them. If a cluster has a distinctive pattern across the top latent dimensions, it is easier to understand why the model assigns samples to that group.


In [ ]:
cluster_profile = build_cluster_profile(
    latent_df,
    cluster_means,
    top_latent_dims,
)

display(cluster_profile.style.format("{:.4f}").background_gradient(cmap="coolwarm", subset=top_latent_dims))


## Step 7 — Ablation: Cluster With Only Selected Latent Dimensions

A feature-importance table is useful, but an ablation test is more concrete.

We rerun KMeans using different slices of the latent space:

- all dimensions
- only the top 2 dimensions
- only the top 3 dimensions
- only the top 5 dimensions
- the weakest 3 dimensions

If the top latent dimensions really carry cluster structure, they should preserve more signal than the weakest dimensions.


In [ ]:
subset_results_df = evaluate_latent_subsets(
    digits_Z_final,
    latent_dim_names,
    ordered_latent_dims,
    true_labels=y_digits,
    n_clusters=N_CLUSTERS,
)

display(subset_results_df.style.format(precision=3).background_gradient(cmap="Greens", axis=0))


In [ ]:
plot_metric_columns = [col for col in ["silhouette", "ARI", "NMI", "matched_accuracy"] if col in subset_results_df.columns]

fig, axes = plt.subplots(1, len(plot_metric_columns), figsize=(5 * len(plot_metric_columns), 4), squeeze=False)
for ax, metric in zip(axes.ravel(), plot_metric_columns):
    sns.barplot(data=subset_results_df, x="subset", y=metric, ax=ax, color="#59A14F")
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## Step 8 — Latent Space Takeaways

This notebook uses DEC as a representation-learning method, not just a source of cluster IDs.

The practical interpretation is:

- A DEC encoder turns images into learned latent coordinates.
- Some coordinates are more cluster-relevant than others.
- Cluster profiles reveal which latent dimensions define each group.
- Ablation tests check whether the strongest coordinates preserve clustering signal.

This is a useful workflow whenever a clustering model produces embeddings: inspect the embedding space directly before treating the final cluster labels as the whole story.


In [ ]:
best_dims = feature_importance.head(3).index.tolist()
weak_dims = feature_importance.tail(3).index.tolist()
dec_metric_row = metrics_df.loc[metrics_df["stage"] == "DEC-refined latent space"].iloc[0]

print("Most useful latent dimensions:", best_dims)
print("Weakest latent dimensions:", weak_dims)
print("Mean DEC confidence:", round(float(latent_confidence.mean()), 4))
print("DEC ARI:", round(float(dec_metric_row["ARI"]), 4))
print("DEC NMI:", round(float(dec_metric_row["NMI"]), 4))
print("Matched cluster accuracy:", round(float(dec_acc), 4))
